# 🔍 01 — Exploratory Data Analysis
**Project:** ISIC 2024 — Skin Cancer Detection with 3D-TBP

**Goal:** Understand the extreme class imbalance, explore the 55 metadata features, and find signals that separate malignant from benign lesions — before touching a model.

---

## 📦 1. Imports & Config

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import cv2
from pathlib import Path

DATA_DIR  = Path('../data/raw')
IMG_DIR   = DATA_DIR / 'train-image' / 'image'

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('tab10')

print('Setup complete.')

## 📄 2. Load Metadata

In [ ]:
df = pd.read_csv(DATA_DIR / 'train-metadata.csv')
print(f'Dataset shape : {df.shape}')
print(f'Columns       : {list(df.columns)}')
print(f'Missing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}')
df.head(3)

## ⚠️ 3. Class Imbalance — The Core Challenge

In [ ]:
counts  = df['target'].value_counts()
n_total = len(df)

print('Class distribution:')
print(f'  Benign    (0): {counts[0]:>7,}  ({counts[0]/n_total*100:.2f}%)')
print(f'  Malignant (1): {counts[1]:>7,}  ({counts[1]/n_total*100:.4f}%)')
print(f'  Imbalance ratio: {counts[0]//counts[1]:,}:1')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar
bars = axes[0].bar(['Benign (0)', 'Malignant (1)'], counts.values,
                    color=['#2196F3', '#F44336'], edgecolor='white', linewidth=1.5, width=0.5)
for bar, count in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000,
                 f'{count:,}', ha='center', va='bottom', fontsize=12, fontweight='bold')
axes[0].set_yscale('log')
axes[0].set_title('Class Distribution (Log Scale)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count (log scale)')

# Pie
axes[1].pie([counts[0], counts[1]],
            labels=[f'Benign\n{counts[0]:,}', f'Malignant\n{counts[1]:,}'],
            colors=['#2196F3', '#F44336'], autopct='%1.3f%%',
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Class Proportion', fontsize=13, fontweight='bold')

plt.suptitle(f'ISIC 2024 — Extreme Class Imbalance ({counts[0]//counts[1]:,}:1 ratio)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 🖼️ 4. Sample Images — Benign vs. Malignant

In [ ]:
n_samples = 5
benign    = df[df['target'] == 0].sample(n_samples, random_state=42)
malignant = df[df['target'] == 1].sample(n_samples, random_state=42)

fig, axes = plt.subplots(2, n_samples, figsize=(20, 8))
labels = {0: ('Benign', '#2196F3'), 1: ('Malignant', '#F44336')}

for col, (_, row) in enumerate(benign.iterrows()):
    path = str(IMG_DIR / f"{row['isic_id']}.jpg")
    img  = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB) if Path(path).exists() else np.zeros((128,128,3), np.uint8)
    axes[0, col].imshow(img); axes[0, col].axis('off')
    if col == 0:
        axes[0, col].set_ylabel('Benign (0)', fontsize=13, fontweight='bold',
                                 color='#2196F3', rotation=0, labelpad=70, va='center')

for col, (_, row) in enumerate(malignant.iterrows()):
    path = str(IMG_DIR / f"{row['isic_id']}.jpg")
    img  = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB) if Path(path).exists() else np.zeros((128,128,3), np.uint8)
    axes[1, col].imshow(img); axes[1, col].axis('off')
    if col == 0:
        axes[1, col].set_ylabel('Malignant (1)', fontsize=13, fontweight='bold',
                                 color='#F44336', rotation=0, labelpad=80, va='center')

plt.suptitle('Sample Lesion Images — Benign vs. Malignant\n(Non-dermoscopic 3D-TBP crops)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

## 📊 5. Metadata Feature Analysis

In [ ]:
# Numerical features: compare distributions by class
num_features = ['age_approx', 'clin_size_long_diam_mm', 'tbp_lv_areaMM2',
                'tbp_lv_norm_color', 'tbp_lv_symm_2axis', 'tbp_lv_color_std_mean']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, feat in zip(axes, num_features):
    if feat not in df.columns:
        ax.text(0.5, 0.5, f'{feat}\nnot found', ha='center', va='center', transform=ax.transAxes)
        continue
    benign_vals    = df[df['target'] == 0][feat].dropna()
    malignant_vals = df[df['target'] == 1][feat].dropna()
    ax.hist(benign_vals,    bins=40, alpha=0.6, color='#2196F3', label='Benign',    density=True)
    ax.hist(malignant_vals, bins=40, alpha=0.8, color='#F44336', label='Malignant', density=True)
    ax.set_title(feat, fontsize=11, fontweight='bold')
    ax.set_xlabel('Value'); ax.set_ylabel('Density')
    ax.legend(fontsize=9)

plt.suptitle('Numerical Feature Distributions: Benign vs. Malignant',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 👤 6. Patient-Level Analysis — Ugly Duckling Principle

In [ ]:
# How many lesions per patient?
patient_lesion_counts = df.groupby('patient_id').size()

print('Lesions per patient:')
print(f'  Mean  : {patient_lesion_counts.mean():.1f}')
print(f'  Median: {patient_lesion_counts.median():.0f}')
print(f'  Max   : {patient_lesion_counts.max()}')
print(f'  Min   : {patient_lesion_counts.min()}')
print(f'  Unique patients: {len(patient_lesion_counts):,}')

# Patients with malignant lesions
mal_patients  = df[df['target'] == 1]['patient_id'].nunique()
total_patients = df['patient_id'].nunique()
print(f'\nPatients with ≥1 malignant lesion: {mal_patients:,} / {total_patients:,} ({mal_patients/total_patients*100:.1f}%)')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.hist(patient_lesion_counts.values, bins=50, color='steelblue', edgecolor='white')
ax1.set_title('Number of Lesions per Patient', fontsize=12, fontweight='bold')
ax1.set_xlabel('Lesion Count per Patient'); ax1.set_ylabel('Number of Patients')
ax1.axvline(patient_lesion_counts.median(), color='red', linestyle='--',
            label=f'Median = {patient_lesion_counts.median():.0f}')
ax1.legend()

# Malignant rate per anatomical site
if 'anatom_site_general' in df.columns:
    site_stats = df.groupby('anatom_site_general')['target'].agg(['mean', 'sum', 'count'])
    site_stats = site_stats.sort_values('mean', ascending=False)
    ax2.barh(site_stats.index, site_stats['mean'] * 100, color='tomato', edgecolor='white')
    ax2.set_title('Malignancy Rate by Anatomical Site', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Malignancy Rate (%)')

plt.tight_layout()
plt.savefig('../results/figures/patient_level_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 🔗 7. Correlation Heatmap (Malignant Subset)

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ['target', 'patient_id']]

# Correlation with target
correlations = df[numeric_cols + ['target']].corr()['target'].drop('target').sort_values(key=abs, ascending=False)

plt.figure(figsize=(10, 8))
colors = ['#F44336' if v > 0 else '#2196F3' for v in correlations.values[:20]]
plt.barh(correlations.index[:20], correlations.values[:20], color=colors, edgecolor='white')
plt.axvline(0, color='black', lw=0.8)
plt.title('Top 20 Features Correlated with Malignancy (target=1)',
          fontsize=13, fontweight='bold')
plt.xlabel('Pearson Correlation with Target')
plt.tight_layout()
plt.savefig('../results/figures/feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 10 most correlated features:')
print(correlations.head(10).to_string())

## ✅ 8. EDA Summary

| Finding | Implication for Modeling |
|---------|-------------------------|
| 1,000:1 class imbalance | Fixed undersampling alone is insufficient — we use dynamic scheduling |
| 55 metadata features, ~30% missing | Imputation + patient-relative feature engineering needed |
| Images are 128×128 low-res TBP crops | No hair removal needed; don't over-resize |
| Patient has multiple lesions | Use Group K-Fold — never let the same patient span train and val |
| Lesion size and color deviate in malignant cases | Patient-relative features capture the ugly duckling signal |

**Next:** `02_Preprocessing.ipynb` — augmentation pipeline, patient-relative feature engineering, fold splits.